# Draft strategy

What roster construction is actually worth in **these two leagues**, as live queries. Re-run the
notebook and the numbers update — nothing below is transcribed by hand.

The question that started this: a "full house" opening — **three running backs and two receivers in
the first five rounds** — on the theory that running back is the scarce position and that even elite
receivers can't carry a team the way an elite back can.

Half of that premise turns out to be true and the conclusion drawn from it doesn't follow. The
scarcity is real, but it is *shallow*: it lives in the first two rounds and is gone by the third,
and a strategy that spends three of five premium picks chasing it is buying the part of the running
back curve that has already flattened. Sections 2-4 measure the premise; section 5 settles the
strategy by simulating ~88,700 drafts off the real historical ADP boards; sections 6-8 are the
honesty checks on that result, one of which changes the recommendation.

**Since this was last run, Sleeper changed the league itself** — 12 teams to 14, and one flex slot
swapped for a dedicated superflex slot — which turns out to matter more than anything else in this
notebook. Every section below is re-run against the league as it now stands; section 1 covers
exactly what changed and section 9 covers what's left of the "what if it were superflex" question
now that one league doesn't have to ask it hypothetically any more.

**Contents**
1. [The two leagues — and the superflex question](#leagues)
2. [Is the premise true? Positional scarcity](#scarcity)
3. [What each round actually returned](#rounds)
4. [Where mid-round running backs go wrong](#hitrate)
5. [The test: simulate the draft](#simulation)
6. [Is any of it significant?](#significance)
7. [Ordering: RB early vs RB often](#ordering)
8. [Who else is at the table](#field)
9. [Superflex: real now, not hypothetical](#superflex)
10. [What to actually do](#doing)

In [1]:
# Find the repo root from wherever the kernel started, so `src` imports work.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy import stats

from src.query import q, tables, columns, peek

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

tables("gold").query("table.str.startswith('draft_strategy')")

,table,layer,rows
6,draft_strategy_results,gold,88704
7,draft_strategy_summary,gold,376


<a id="leagues"></a>
## 1. The two leagues — and the superflex question

**Sleeper is now superflex. ESPN still isn't.** That split didn't exist when this notebook was
first written — both leagues started exactly one quarterback, and section 9 below was a pure
"what if" exercise. It stopped being one for Sleeper mid-August: `league_settings`, which reads
each platform's own settings rather than anything typed in by hand, now shows a `SUPER_FLEX` entry
in Sleeper's `roster_positions` and 14 teams where there used to be 12. ESPN is unchanged — its
lineup slot 7 (`OP`, its superflex slot) is still zero.

Every table from here on runs against Sleeper's *current* settings, so the superflex effect is
already baked into everything in sections 2-8, not held back for a special section. Section 9 is
what's left of the "what if" question: for ESPN it's still the original hypothetical, and for
Sleeper it now asks something narrower — what a *second* superflex slot would do on top of the one
it already has.

The two leagues differ in more ways than the format label suggests, and the differences don't all
push the same direction:

- **Sleeper** is 14 teams, half-PPR, with **one** flex and **one** superflex slot — 8 skill
  starters per team (unchanged: a flex slot became a superflex slot, not an extra one), so 112
  skill players start every week, up from 96 before the change.
- **ESPN** is 10 teams, full PPR, with **one** flex and no superflex — 7 skill starters, so only 70
  start.

More teams make the pool thinner on their own; full PPR lifts receivers; and now one league also
asks its teams to start two quarterbacks-or-better every week. That last one is not a small effect
— section 6 is where it turns out to be the largest number in this notebook.

One caveat belongs here rather than buried in section 9, because it now bears on the *actual*
numbers, not just a hypothetical: every simulated draft below still drafts off `adp_consensus`,
which is FantasyPros/FFC consensus ADP — a market priced for the one-quarterback leagues most of
its users play, not for this one. A real draft room that has fully absorbed "this league is
superflex now" would price quarterbacks higher than that board does. What follows says what a
superflex lineup was worth while quarterbacks were still available at one-quarterback prices —
close to true for a league's first draft under new rules, and less true every year after.

In [2]:
q("""
    SELECT
        league_key                          AS league,
        team_count                          AS teams,
        rec_pts                             AS "pts/reception",
        qb_slots || ' QB'                   AS qb,
        rb_slots || ' RB'                   AS rb,
        wr_slots || ' WR'                   AS wr,
        te_slots || ' TE'                   AS te,
        flex_slots || ' FLEX'               AS flex,
        superflex_slots                     AS "superflex slots",
        bench_slots                         AS bench,
        qb_slots + rb_slots + wr_slots + te_slots + flex_slots + superflex_slots
                                            AS "skill starters",
        team_count * (qb_slots + rb_slots + wr_slots + te_slots + flex_slots + superflex_slots)
                                            AS "skill players starting"
    FROM league_settings
    ORDER BY league_key
""").set_index("league").T

league,espn,sleeper
teams,10,14
pts/reception,1.0,0.5
qb,1 QB,1 QB
rb,2 RB,2 RB
wr,2 WR,2 WR
te,1 TE,1 TE
flex,1 FLEX,1 FLEX
superflex slots,0,1
bench,7,5
skill starters,7,8


In [3]:
# The superflex claim, checked rather than asserted.
sf = q("SELECT league_key, superflex_slots, qb_slots FROM league_settings ORDER BY league_key")
for _, row in sf.iterrows():
    verdict = (
        f"{row.superflex_slots} superflex slot(s) — QB is flex-eligible"
        if row.superflex_slots
        else f"no superflex slot: {row.qb_slots} starting QB, and QB cannot fill a flex spot"
    )
    print(f"{row.league_key:>8}: {verdict}")

print(
    "\nsuperflex leagues among mine:",
    int((sf.superflex_slots > 0).sum()), "of", len(sf),
)

    espn: no superflex slot: 1 starting QB, and QB cannot fill a flex spot
 sleeper: 1 superflex slot(s) — QB is flex-eligible

superflex leagues among mine: 1 of 2


<a id="scarcity"></a>
## 2. Is the premise true? Positional scarcity

**Partly.** An elite running back really is worth more than an elite receiver — and the gap closes
fast.

The table below is points over replacement by *realised* positional finish, averaged over 2015-2025,
in each league's own scoring. `RB-WR` is the whole argument in one column: how much more the RB
finishing Nth was worth than the WR finishing Nth.

Read where that column crosses zero. Above the crossover, running back is the scarcer asset and the
"full house" premise holds. Below it, the premise inverts and receivers are worth *more* at the same
rank — because so many more of them start (`starters_at_position` in `points_over_replacement`
counts dedicated slots plus the flex spots that position actually won), which pushes the receiver
replacement level far deeper down the list.

The crossover lands in a very different place in each league, and that difference is most of why
these two leagues want different drafts. Sleeper's crossover moved from rank 15 to rank 18 the
moment its superflex slot went live: quarterbacks now compete for the same shared flex-eligible pool
as RB/WR/TE (`points_over_replacement.py`'s replacement-level calculation was updated to match
`draft_strategy.py`'s own lineup logic), which pulls a few more marginal starts away from running
back and pushes its replacement level — and the crossover — a little deeper.

The table also carries a QB column, mostly as a control for this section's RB-vs-WR question — but
in Sleeper it's no longer a quiet one. With quarterbacks now sharing that pool, Sleeper's QB1 is
worth +280 points over replacement, more than its RB1. That's a different finding than the one this
section is making, and it belongs to section 6, not here.

In [4]:
def por_curve(league_key, max_rank=30):
    return q("""
        SELECT position_rank AS rank,
               MAX(CASE WHEN position = 'RB' THEN por END) AS RB,
               MAX(CASE WHEN position = 'WR' THEN por END) AS WR,
               MAX(CASE WHEN position = 'TE' THEN por END) AS TE,
               MAX(CASE WHEN position = 'QB' THEN por END) AS QB
        FROM (
            SELECT position, position_rank, AVG(points_over_replacement) AS por
            FROM points_over_replacement
            WHERE league_key = ? AND season <= (SELECT MAX(season) FROM weekly_stats)
            GROUP BY position, position_rank
        )
        GROUP BY position_rank
        HAVING position_rank <= ?
        ORDER BY position_rank
    """, [league_key, max_rank])


curves = {}
for league_key in ("sleeper", "espn"):
    curve = por_curve(league_key).set_index("rank")
    curve["RB-WR"] = curve.RB - curve.WR
    curves[league_key] = curve

pd.concat(curves, axis=1).round(1)

sleeper                              espn                           
          RB     WR     TE     QB RB-WR     RB     WR     TE     QB RB-WR
rank                                                                     
1      216.4  171.5  117.7  280.2  44.9  205.3  181.8  121.7  114.3  23.5
2      174.4  144.6   85.9  241.7  29.9  153.1  145.4   86.9   76.0   7.7
3      158.0  127.2   69.1  232.1  30.8  137.4  130.8   63.4   67.0   6.5
4      144.4  113.4   60.0  221.3  31.0  119.2  114.4   51.2   54.3   4.9
5      126.7  106.5   50.4  206.2  20.2  102.6  103.4   40.2   39.0  -0.8
6      116.7   97.3   42.0  192.0  19.4   92.5   94.5   31.0   26.7  -2.0
7      104.0   86.9   34.1  185.6  17.1   75.1   82.8   18.4   20.7  -7.8
8       93.7   80.9   28.9  181.0  12.8   66.0   75.3   14.6   15.4  -9.3
9       85.5   76.9   25.4  175.4   8.6   58.9   69.2   10.6   11.0 -10.3
10      81.6   72.5   19.5  171.3   9.1   53.5   63.0    3.3    5.5  -9.5
11      76.8   67.9   13.2  166.0   8.9   47.0   60.3    0.0    0.0 -13.3
12      70.6   65.6   10.2  156.4   5.0   41.9   56.4   -5.3   -9.9 -14.5
13      67.4   63.1    7.8  146.1   4.3   36.3   52.2  -10.4  -22.4 -15.8
14      64.4   60.9    3.8  140.1   3.5   32.2   49.4  -16.7  -25.5 -17.2
15      58.6   54.9    0.0  132.5   3.7   27.5   46.2  -20.0  -32.8 -18.7
16      54.1   51.1   -3.0  126.9   3.0   24.6   40.1  -24.3  -37.5 -15.5
17      50.5   48.2   -6.0  120.7   2.3   20.6   36.4  -27.0  -45.5 -15.8
18      47.2   46.6   -7.7  112.7   0.7   15.5   32.9  -30.3  -53.9 -17.4
19      44.3   44.4  -10.4  109.3  -0.1   12.3   30.5  -34.1  -58.8 -18.2
20      40.5   41.2  -13.6  101.5  -0.7    8.9   28.0  -36.5  -64.4 -19.1
21      37.5   39.8  -17.8   91.7  -2.3    5.5   26.6  -40.6  -73.8 -21.0
22      34.7   38.3  -19.8   82.0  -3.7    0.5   22.9  -42.9  -83.9 -22.3
23      30.6   36.0  -21.8   73.7  -5.4   -3.0   19.2  -47.2  -92.2 -22.2
24      25.9   32.6  -25.0   67.6  -6.7   -7.8   17.4  -50.2  -97.4 -25.1
25      21.4   29.9  -27.4   57.9  -8.5  -11.0   13.4  -54.4 -107.7 -24.5
26      17.5   27.1  -30.7   50.8  -9.6  -14.6   10.8  -58.7 -112.7 -25.4
27      15.2   24.5  -33.6   43.6  -9.3  -17.5    7.4  -62.3 -122.3 -24.9
28      11.0   22.7  -36.1   32.6 -11.8  -20.9    4.8  -64.4 -132.2 -25.6
29       7.2   20.3  -38.1   20.3 -13.2  -26.1    2.2  -65.7 -143.5 -28.3
30       5.5   19.5  -40.2    7.4 -14.0  -28.8    0.2  -68.1 -157.0 -29.0

In [5]:
# Where does the running back premium run out? First rank at which the WR finishing there is worth
# more than the RB finishing there.
for league_key, curve in curves.items():
    ahead = curve.index[curve["RB-WR"] > 0]
    crossover = int(ahead.max()) + 1 if len(ahead) else 1
    top2 = curve.loc[:2, "RB-WR"].mean()
    print(
        f"{league_key:>8}: RB is worth more than WR through rank {crossover - 1}, "
        f"WR from rank {crossover} on. "
        f"Premium at the top (ranks 1-2): {top2:+.0f} pts."
    )

 sleeper: RB is worth more than WR through rank 18, WR from rank 19 on. Premium at the top (ranks 1-2): +37 pts.
    espn: RB is worth more than WR through rank 4, WR from rank 5 on. Premium at the top (ranks 1-2): +16 pts.


<a id="rounds"></a>
## 3. What each round actually returned

Section 2 measured finishes. This one measures **picks**, which is the thing a strategy actually
controls: for every player with a preseason consensus ADP since 2015, what did a pick in that round
go on to return? Value is `draft_value.actual_value` — points over replacement in that league's
scoring, floored at zero, because nobody is forced to start a player worse than the waiver wire — and
a drafted player who never played a snap is carried as a genuine zero rather than dropped, so the
bust rate is priced in.

Rounds are that league's own team count, so "round 3" means picks 29-42 in the now-14-team Sleeper
league and 21-30 in ESPN.

Running back leads receiver in most of the eight-round window in Sleeper — every round but the
fourth and eighth. In full-PPR ESPN receiver leads from the very first round; running back never
takes the lead there.

Rounds 3-5 are exactly where a "full house" opening spends its third running back.

In [6]:
def value_by_round(league_key, rounds=8):
    return q("""
        SELECT CAST(CEIL(d.consensus_adp / l.team_count) AS INT) AS round,
               AVG(CASE WHEN position = 'RB' THEN actual_value END) AS RB,
               AVG(CASE WHEN position = 'WR' THEN actual_value END) AS WR,
               AVG(CASE WHEN position = 'TE' THEN actual_value END) AS TE,
               AVG(CASE WHEN position = 'QB' THEN actual_value END) AS QB,
               COUNT(*) AS picks
        FROM draft_value d
        JOIN league_settings l ON l.league_key = d.league_key
        WHERE d.league_key = ? AND d.actual_value IS NOT NULL
          AND d.consensus_adp <= l.team_count * ?
        GROUP BY 1 ORDER BY 1
    """, [league_key, rounds])


returns = {k: value_by_round(k).set_index("round") for k in ("sleeper", "espn")}
pd.concat(returns, axis=1).round(1)

sleeper                           espn                         
           RB    WR    TE     QB picks    RB    WR     TE    QB picks
round                                                                
1        94.1  85.4  70.5   20.1   146  79.8  80.0   83.9   NaN    98
2        64.8  54.9  63.1  197.3   153  61.5  72.0  105.6  25.2   121
3        45.1  44.8  61.6  176.5   165  33.4  38.5   43.2  62.0    99
4        25.4  42.2  39.8  149.0   143  23.9  37.6   55.2  35.2   116
5        28.6  25.1  25.8  174.1   154  16.8  37.2   33.2  22.2   115
6        21.8  17.1  14.5  132.6   153  16.0  23.8   22.6  27.5   105
7        16.8  16.6  10.5  129.9   151  12.6  23.3   24.7  28.5   107
8        13.5  14.3  18.1  152.8   147  10.1  12.1    7.0   8.1   111

In [7]:
# Which position returned more, per round, per league.
for league_key, table in returns.items():
    rb_rounds = table.index[table.RB > table.WR].tolist()
    print(f"{league_key:>8}: RB out-returned WR in round(s) {rb_rounds or 'none'} "
          f"of the first {len(table)}")

 sleeper: RB out-returned WR in round(s) [1, 2, 3, 5, 6, 7] of the first 8
    espn: RB out-returned WR in round(s) none of the first 8


<a id="hitrate"></a>
## 4. Where mid-round running backs go wrong

The averages above hide *how* they happen, and the how is the useful part. `boom_bust` classifies
every drafted player-season by whether he finished as a top-`team_count` asset at his position — an
absolute measure, not "did he beat his ADP" — so this is the hit rate on a pick.

Rounds 1-3 look similar for both positions. The separation is in **rounds 4-5**: a running back
taken there hits far less often than a receiver taken there, in both leagues. That is the specific
failure mode of the full-house opening. It isn't that the third running back is a bad player, it's
that the fourth- and fifth-round running back is the least reliable pick on the board, and the
strategy commits to taking one before the draft starts.

`n` columns are shown because the tight end row is thin — few tight ends go early enough to appear —
and shouldn't be read as a finding.

In [8]:
def hit_rate(league_key, rounds=6):
    return q("""
        SELECT CAST(CEIL(b.consensus_adp / l.team_count) AS INT) AS round,
               100 * AVG(CASE WHEN position = 'RB' THEN is_elite_finish::INT END) AS "RB hit%",
               100 * AVG(CASE WHEN position = 'WR' THEN is_elite_finish::INT END) AS "WR hit%",
               SUM(CASE WHEN position = 'RB' THEN 1 ELSE 0 END) AS "n RB",
               SUM(CASE WHEN position = 'WR' THEN 1 ELSE 0 END) AS "n WR",
               100 * AVG(CASE WHEN position = 'TE' THEN is_elite_finish::INT END) AS "TE hit%",
               100 * AVG(CASE WHEN position = 'QB' THEN is_elite_finish::INT END) AS "QB hit%"
        FROM boom_bust b
        JOIN league_settings l ON l.league_key = b.league_key
        WHERE b.league_key = ? AND b.consensus_adp <= l.team_count * ?
        GROUP BY 1 ORDER BY 1
    """, [league_key, rounds])


hits = {k: hit_rate(k).set_index("round") for k in ("sleeper", "espn")}
pd.concat(hits, axis=1).round(1)

sleeper                                        espn                                    
      RB hit% WR hit%  n RB  n WR TE hit% QB hit% RB hit% WR hit%  n RB  n WR TE hit% QB hit%
round                                                                                        
1        65.1    64.8  86.0  54.0    75.0     0.0    52.5    55.9  61.0  34.0   100.0     NaN
2        50.0    50.0  58.0  68.0    80.0    81.2    46.3    51.9  54.0  52.0    83.3    75.0
3        37.3    33.3  59.0  72.0    86.7    80.0    33.3    20.5  36.0  44.0    50.0    77.8
4         9.8    33.3  41.0  63.0    64.7    59.1    25.0    24.0  40.0  50.0    77.8    61.5
5        18.2    18.6  55.0  59.0    52.9    66.7    10.8    25.9  37.0  54.0    66.7    41.7
6        12.2    10.2  49.0  59.0    55.0    47.6    10.8    13.2  37.0  38.0    36.4    55.6

In [9]:
for league_key, table in hits.items():
    late = table.loc[4:5]
    print(f"{league_key:>8}: rounds 4-5 hit rate — RB {late['RB hit%'].mean():.1f}% "
          f"vs WR {late['WR hit%'].mean():.1f}%")

 sleeper: rounds 4-5 hit rate — RB 14.0% vs WR 26.0%
    espn: rounds 4-5 hit rate — RB 17.9% vs WR 25.0%


<a id="simulation"></a>
## 5. The test: simulate the draft

Everything above is circumstantial. A draft strategy is a claim about *roster construction*, and the
only way to settle it is to run the draft — so `src/gold/draft_strategy.py` does, ~88,700 times.

Each simulated draft is a snake draft off that season's real `adp_consensus` board, with every team
scored on the hindsight-best starting lineup it could field from the roster it ended up with, using
that league's own scoring. One team — the focal team — follows a strategy for the first five rounds;
everyone else takes the best player left by ADP. Then the focal seat rotates through every draft
slot, and the whole thing repeats for every season from 2015 on.

A **strategy** here is a *composition*: how many of each position in the first five rounds, with ADP
deciding the order. That is what a human actually does — nobody committed to "3 RB, 2 WR" passes the
top receiver on the board at 1.03 to force a back — and it keeps the strategy space at 36 instead of
4⁵ = 1024. `ADP` is the control: no constraint at all, best available for all fifteen-ish rounds.

`points_vs_field` is the focal team's starting-lineup total minus the average of the other teams in
that same draft, so the control sits at exactly zero by construction. Full details and caveats are
in the module docstring.

In [10]:
q("""
    SELECT league_key AS league, variant, field_model AS field,
           COUNT(*) AS drafts,
           COUNT(DISTINCT strategy) AS strategies,
           COUNT(DISTINCT season) AS seasons,
           MIN(season) AS "from", MAX(season) AS "to"
    FROM draft_strategy_results
    GROUP BY ALL ORDER BY league, variant, field
""")

,league,variant,field,drafts,strategies,seasons,from,to
0,espn,actual,adp,6270,57,11,2015,2025
1,espn,actual,mixed,12210,37,11,2015,2025
2,espn,superflex,adp,6270,57,11,2015,2025
3,espn,superflex,mixed,12210,37,11,2015,2025
4,sleeper,actual,adp,8778,57,11,2015,2025
5,sleeper,actual,mixed,17094,37,11,2015,2025
6,sleeper,superflex,adp,8778,57,11,2015,2025
7,sleeper,superflex,mixed,17094,37,11,2015,2025


In [11]:
def ranking(league_key, variant="actual", field_model="adp"):
    table = q("""
        SELECT strategy, points_vs_field, finish_rank, 100 * win_rate AS "win%",
               100 * top_third_rate AS "top3rd%", t_stat, seasons_positive, n_seasons
        FROM draft_strategy_summary
        WHERE league_key = ? AND variant = ? AND field_model = ?
          AND strategy_kind <> 'ordering'
        ORDER BY points_vs_field DESC
    """, [league_key, variant, field_model]).reset_index(drop=True)
    table.index = range(1, len(table) + 1)
    return table


for league_key in ("sleeper", "espn"):
    table = ranking(league_key)
    keep = pd.concat([
        table.head(5),
        table[table.strategy.isin(["ADP", "3RB2WR", "2RB3WR"])],
        table.tail(3),
    ]).drop_duplicates("strategy").sort_values("points_vs_field", ascending=False)
    print(f"\n=== {league_key} — opening composition, best to worst (of {len(table)}) ===")
    print(keep.round(2).to_string())


=== sleeper — opening composition, best to worst (of 37) ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1      2QB1RB2WR            85.38         5.81   8.44    55.84    5.75                11         11
2      2QB2RB1WR            82.73         5.75  10.39    57.14    5.56                11         11
3   2QB1RB1WR1TE            77.44         5.86  10.39    53.90    4.40                10         11
4      2QB2RB1TE            64.44         6.05  12.34    55.19    2.80                 8         11
5      2QB1RB2TE            58.66         6.30   6.49    44.81    3.17                 8         11
22           ADP            -0.00         7.50   7.14    35.71   -2.83                 3         11
23        2RB3WR           -28.09         7.96   4.55    28.57   -2.46                 5         11
28        3RB2WR           -38.45         8.23   2.60    25.32   -3.36                 1         11
35           5RB           -74.99     


=== espn — opening composition, best to worst (of 37) ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1   1QB1RB2WR1TE            17.68         5.08  15.45    48.18    1.21                 6         11
2      1RB2WR2TE            14.50         5.11  15.45    47.27    0.87                 6         11
3         1RB4WR             9.28         5.49  11.82    40.91    0.64                 5         11
4      1RB3WR1TE             9.00         5.47  15.45    41.82    0.71                 6         11
5      2RB2WR1TE             8.40         5.49  15.45    43.64    0.86                 7         11
7         2RB3WR             5.92         5.60   8.18    39.09    0.71                 7         11
12           ADP             0.00         5.50  10.00    40.00    0.18                 5         11
18        3RB2WR            -9.90         5.77   4.55    30.91   -1.15                 6         11
35        4RB1TE           -45.66        

In [12]:
# Where the full house actually lands, and what the control would have paid instead.
for league_key in ("sleeper", "espn"):
    table = ranking(league_key)
    place = {s: int(table.index[table.strategy == s][0]) for s in ("3RB2WR", "ADP")}
    full_house = table.loc[place["3RB2WR"]]
    print(
        f"{league_key:>8}: 3RB2WR ranks {place['3RB2WR']}/{len(table)} "
        f"({full_house.points_vs_field:+.1f} pts vs field, "
        f"{full_house.seasons_positive}/{full_house.n_seasons} seasons positive) — "
        f"the do-nothing ADP control ranks {place['ADP']}."
    )

 sleeper: 3RB2WR ranks 28/37 (-38.5 pts vs field, 1/11 seasons positive) — the do-nothing ADP control ranks 22.
    espn: 3RB2WR ranks 18/37 (-9.9 pts vs field, 6/11 seasons positive) — the do-nothing ADP control ranks 12.


<a id="significance"></a>
## 6. Is any of it significant?

This is the section that keeps section 5 honest, and this time it pulls the conclusion in two
different directions for the two leagues.

Thirty-seven strategies were ranked per league. At the usual threshold roughly two of them would
clear |t| > 2 by chance alone, so picking the top row and calling it the best strategy is exactly
the mistake sampling noise is designed to produce. And the t-tests below are already the
*conservative* version: they run on the eleven per-season means rather than the individual drafts
within a season, because drafts within one season share the same player outcomes — one running back
tearing an ACL moves every RB-heavy draft that year together — and treating those as independent
would shrink the standard error by roughly a factor of the number of draft slots and make almost
everything "significant".

Instead of trusting any single strategy, the test below asks a **trend** question, which has far
more power: holding everything else loose, does adding one more of a position to the opening help
or hurt? Each cell is the mean `points_vs_field` across every strategy with that many of that
position; the slope is fitted per season and t-tested across seasons.

**In ESPN, nothing has changed: no position's slope clears significance** — not running back, not
receiver, not tight end, and quarterback least of all (t = 0.06, the flattest number in this table).
Zero of its 37 actual-settings strategies clear |t| > 2 on the positive side; the only significant
openings are the extreme, all-in ones on the losing side.

**In Sleeper, one thing swamps everything else: quarterback.** Each additional quarterback in the
opening is worth **+59 points per pick, t = 7.36** — an order of magnitude past the |t| > 2 bar, and
by a wide margin the largest, most significant number this notebook produces anywhere. Running back
and receiver still show the same shallow, inverted-U shape as before — a small early gain, worse
from the fourth pick on, neither slope clearing significance on its own — that part of the finding
is unchanged. What's changed is that it no longer matters much: 14 of Sleeper's 37 strategies now
clear |t| > 2 on the positive side, against ~2 expected by chance, and every one of them opens with
two quarterbacks. None of the RB/WR-flavored openings that used to carry this section —
`2RB2WR1TE` among them — are significant any more; `2RB2WR1TE` is now on the *bad* list.

That also kills the one cross-league finding this section used to have. The strategy that used to
clear significance in both leagues at once, `2RB2WR1TE`, doesn't exist any more — with Sleeper's
board now organized entirely around quarterback count and ESPN's not organized around anything,
**there is no opening that is significantly good in both leagues at once.** Whatever you draft, it
has to be chosen per league now, not carried over as one shared plan.

In [13]:
compositions = q("""
    SELECT * FROM draft_strategy_results
    WHERE strategy_kind = 'composition' AND field_model = 'adp'
""")
for position in ("QB", "RB", "WR", "TE"):
    compositions[position] = (
        compositions.strategy.str.extract(rf"(\d){position}").fillna(0).astype(int)
    )


def trend(frame, position):
    """Mean vs-field by count, plus a season-clustered test of the slope."""
    per_season = frame.groupby(["season", position]).points_vs_field.mean().reset_index()
    slopes = [
        np.polyfit(group[position], group.points_vs_field, 1)[0]
        for _, group in per_season.groupby("season")
        if group[position].nunique() > 1
    ]
    t_stat, p_value = stats.ttest_1samp(slopes, 0.0)
    cells = frame.groupby(position).points_vs_field.mean()
    return {
        "position": position,
        **{f"{n} in opening": cells.get(n, np.nan) for n in range(6)},
        "pts/extra pick": np.mean(slopes),
        "t": t_stat,
        "p": p_value,
    }


for league_key in ("sleeper", "espn"):
    frame = compositions[
        (compositions.league_key == league_key) & (compositions.variant == "actual")
    ]
    print(f"\n=== {league_key} / actual — value of the Nth pick spent on a position ===")
    print(pd.DataFrame([trend(frame, p) for p in ("QB", "RB", "WR", "TE")])
          .set_index("position").round(2).to_string())


=== sleeper / actual — value of the Nth pick spent on a position ===
          0 in opening  1 in opening  2 in opening  3 in opening  4 in opening  5 in opening  pts/extra pick     t     p
position                                                                                                                
QB              -55.13         26.31         63.26           NaN           NaN           NaN           59.20  7.36  0.00
RB               -8.79         25.79         17.04         -6.28        -39.52        -74.99          -15.72 -2.07  0.07
WR                0.51         17.70         14.95         -8.07        -36.50        -67.49          -15.02 -2.05  0.07
TE                7.66          5.89        -14.17           NaN           NaN           NaN          -10.91 -1.77  0.11

=== espn / actual — value of the Nth pick spent on a position ===
          0 in opening  1 in opening  2 in opening  3 in opening  4 in opening  5 in opening  pts/extra pick     t     p
position        

In [14]:
# Which strategies clear |t| > 2, and which way do they point?
significant = q("""
    SELECT league_key AS league, variant,
           COUNT(*) AS strategies,
           ROUND(0.05 * COUNT(*), 1) AS "expected by chance",
           SUM((ABS(t_stat) > 2)::INT) AS "|t|>2",
           SUM((t_stat > 2)::INT) AS good,
           SUM((t_stat < -2)::INT) AS bad
    FROM draft_strategy_summary
    WHERE field_model = 'adp' AND strategy_kind <> 'ordering'
    GROUP BY ALL ORDER BY league, variant
""")
print(significant.to_string(index=False))

named = q("""
    SELECT league_key, variant, t_stat > 2 AS good, strategy
    FROM draft_strategy_summary
    WHERE field_model = 'adp' AND strategy_kind <> 'ordering' AND ABS(t_stat) > 2
    ORDER BY league_key, variant, t_stat DESC
""")
for (league_key, variant), group in named.groupby(["league_key", "variant"]):
    for good in (True, False):
        names = group[group.good == good].strategy.tolist()
        if names:
            print(f"\n{league_key:>8} / {variant:<9} significantly "
                  f"{'GOOD' if good else 'BAD ':<4}: {', '.join(names)}")

 league   variant  strategies  expected by chance  |t|>2  good  bad
   espn    actual          37                 1.9    6.0   0.0  6.0
   espn superflex          37                 1.9   14.0   7.0  7.0
sleeper    actual          37                 1.9   29.0  14.0 15.0
sleeper superflex          37                 1.9   28.0  14.0 14.0



    espn / actual    significantly BAD : 2QB1RB2TE, 4RB1TE, 5RB, 1QB4RB, 4RB1WR, 2QB3RB

    espn / superflex significantly GOOD: 2QB1RB1WR1TE, 2QB1RB2WR, 1QB1RB3WR, 1QB1RB2WR1TE, 2QB2RB1WR, 1QB2RB2WR, 2QB2WR1TE

    espn / superflex significantly BAD : 3RB2TE, 5RB, 4RB1TE, 2RB2WR1TE, 4RB1WR, 3RB2WR, 3RB1WR1TE

 sleeper / actual    significantly GOOD: 2QB1RB2WR, 2QB2RB1WR, 1QB1RB2WR1TE, 1QB2RB2WR, 2QB1RB1WR1TE, 1QB1RB3WR, 1QB2RB1WR1TE, 1QB3RB1WR, 2QB1RB2TE, 2QB2RB1TE, 2QB2WR1TE, 2QB3RB, 2QB1WR2TE, 2QB3WR

 sleeper / actual    significantly BAD : 2RB3WR, 4RB1TE, 1RB3WR1TE, ADP, 3RB1WR1TE, 2RB2WR1TE, 5WR, 5RB, 3RB2WR, 4WR1TE, 3RB2TE, 4RB1WR, 2RB1WR2TE, 3WR2TE, 1RB2WR2TE

 sleeper / superflex significantly GOOD: 2QB1RB2WR, 2QB2RB1WR, 2QB1RB2TE, 2QB1RB1WR1TE, 2QB2WR1TE, 2QB3WR, 2QB1WR2TE, 2QB3RB, 2QB2RB1TE, 1QB2RB2WR, 1QB1RB3WR, 1QB1RB2WR1TE, 1QB3RB1WR, 1QB2RB1WR1TE

 sleeper / superflex significantly BAD : 5WR, 4RB1TE, 5RB, 4WR1TE, 2RB3WR, 1RB3WR1TE, 3RB2TE, 3RB1WR1TE, 4RB1WR, 3RB2WR, 2R

In [15]:
# The only claim that survives being asked of both leagues at once. Every league key has to
# contribute its own set explicitly -- one league can come up with zero significant strategies
# (ESPN does, under the current settings), and grouping only the rows that survived would silently
# drop it from the intersection instead of correctly zeroing it out.
actual = named[(named.variant == "actual") & named.good]
league_keys = sorted(compositions.league_key.unique())
in_both = set.intersection(*(
    set(actual[actual.league_key == lk].strategy) for lk in league_keys
))
print("openings significantly better than the field in BOTH leagues:", ", ".join(sorted(in_both)) or "none")

q("""
    SELECT league_key AS league, strategy, points_vs_field, finish_rank,
           100 * top_third_rate AS "top3rd%", t_stat, seasons_positive, n_seasons
    FROM draft_strategy_summary
    WHERE variant = 'actual' AND field_model = 'adp' AND strategy = ?
    ORDER BY league_key
""", [sorted(in_both)[0]]).round(2) if in_both else None

openings significantly better than the field in BOTH leagues: none


<a id="ordering"></a>
## 7. Ordering: RB early vs RB often

If composition barely matters — still true in ESPN, no longer the main story in Sleeper — does the
*order* within a composition matter, for the RB/WR-only openings that used to be this notebook's
main event?

On average, yes, in both leagues: opening with a running back beats opening with two receivers by
+7.5 points in Sleeper and +2.5 in ESPN. But the strict version of that claim — *every* RB-first
sequence beats *every* WR-WR-first sequence — no longer holds in either league, and it used to hold
in both, without exception, the last time this notebook was run. The worst RB-opener
(`RB-WR-RB-RB-WR`) now trails the best WR-WR-opener in both leagues: -47.5 vs -44.4 in Sleeper, and
-8.6 vs +7.6 in ESPN (that ESPN "WR-WR" leader, `RB-WR-WR-RB-WR`, actually opens with a back —
exactly the kind of overlap that breaks the "every/every" framing). The average edge survives; the
universal one doesn't.

Read that as a downgrade in confidence, not a reversal: the mean still favors RB-first in both
leagues, by about the same margin as before. What's gone is the stronger claim that it's *safe* —
with only ten sequences a side, one particular player landing in one particular slot is enough to
flip the extremes. That it flipped in ESPN too, where nothing about the league's own settings
changed, points at the most likely cause: this rebuild also picked up newer ADP rows (see the
ADP-loader fix in the git log) on top of the Sleeper settings change, and the ordering test is far
more sensitive to exactly which players anchor each rank than the composition test in section 6 is.

The bigger point is unchanged: for either league, the ordering effect (28-44 points end to end) is
worth less than getting the *composition* right (43-91 points across the sensible openings) — and
in Sleeper, worth far less than getting the *quarterback count* right (dozens to hundreds of points,
§6).

In [16]:
orderings = q("""
    SELECT league_key, strategy, points_vs_field, finish_rank, t_stat,
           seasons_positive, n_seasons
    FROM draft_strategy_summary
    WHERE strategy_kind = 'ordering' AND variant = 'actual' AND field_model = 'adp'
    ORDER BY league_key, points_vs_field DESC
""")
orderings["opens"] = orderings.strategy.str.split("-").str[0]
orderings["RBs"] = orderings.strategy.str.count("RB")

for league_key in ("sleeper", "espn"):
    table = orderings[orderings.league_key == league_key].drop(columns="league_key")
    print(f"\n=== {league_key} — forced opening sequences, best to worst ===")
    print(table.round(2).to_string(index=False))


=== sleeper — forced opening sequences, best to worst ===
      strategy  points_vs_field  finish_rank  t_stat  seasons_positive  n_seasons opens  RBs
RB-RB-WR-WR-WR            -8.98         7.72   -0.46                 4         11    RB    2
WR-RB-RB-WR-WR           -21.31         7.84   -0.96                 5         11    WR    2
RB-RB-RB-WR-WR           -23.11         7.98   -1.16                 2         11    RB    3
RB-RB-WR-WR-RB           -24.54         8.07   -1.34                 4         11    RB    3
WR-RB-WR-WR-RB           -27.85         8.07   -1.13                 5         11    WR    2
RB-RB-WR-RB-WR           -28.72         8.12   -1.16                 2         11    RB    3
WR-RB-WR-RB-WR           -28.73         8.00   -1.47                 2         11    WR    2
RB-WR-RB-WR-WR           -29.35         8.01   -2.24                 2         11    RB    2
RB-WR-WR-WR-RB           -32.10         8.14   -2.45                 1         11    RB    2
WR-RB-RB-WR

In [17]:
# The grouped claim: does opening with a back beat opening with a receiver?
rows = []
for league_key in ("sleeper", "espn"):
    table = orderings[orderings.league_key == league_key]
    rb_first = table[table.opens == "RB"].points_vs_field
    wr_first = table[table.opens == "WR"].points_vs_field
    three_rb = table[table.RBs == 3].points_vs_field
    two_rb = table[table.RBs == 2].points_vs_field
    rows.append({
        "league": league_key,
        "opens RB": rb_first.mean(),
        "opens WR": wr_first.mean(),
        "RB-first edge": rb_first.mean() - wr_first.mean(),
        "3 RB total": three_rb.mean(),
        "2 RB total": two_rb.mean(),
        "3-RB edge": three_rb.mean() - two_rb.mean(),
    })
pd.DataFrame(rows).set_index("league").round(1)

,opens RB,opens WR,RB-first edge,3 RB total,2 RB total,3-RB edge
league,,,,,,
sleeper,-31.1,-38.5,7.5,-36.5,-33.2,-3.3
espn,3.5,0.9,2.5,-0.9,5.3,-6.3


In [18]:
# "Every RB opener beats every WR-WR opener", checked. Also: how big is the ordering effect next to
# the composition effect it sits inside?
for league_key in ("sleeper", "espn"):
    table = orderings[orderings.league_key == league_key]
    rb_open = table[table.opens == "RB"].points_vs_field
    wr_wr_open = table[table.strategy.str.startswith("WR-WR")].points_vs_field
    holds = rb_open.min() > wr_wr_open.max()

    # "Sensible" = no all-in openings and no doubling up at QB or TE, i.e. the compositions a
    # human would actually be choosing between.
    all_compositions = ranking(league_key)
    sensible = all_compositions[
        ~all_compositions.strategy.str.contains(r"[45](?:RB|WR)|2QB|2TE")
    ].points_vs_field
    print(
        f"{league_key:>8}: worst RB-opener {rb_open.min():+.1f} vs best WR-WR opener "
        f"{wr_wr_open.max():+.1f} -> claim holds: {holds}\n"
        f"{'':>10}ordering spread {table.points_vs_field.max() - table.points_vs_field.min():.0f} pts; "
        f"spread across sensible compositions {sensible.max() - sensible.min():.0f} pts"
    )

 sleeper: worst RB-opener -47.5 vs best WR-WR opener -44.4 -> claim holds: False
          ordering spread 44 pts; spread across sensible compositions 91 pts
    espn: worst RB-opener -8.6 vs best WR-WR opener +7.6 -> claim holds: False
          ordering spread 28 pts; spread across sensible compositions 43 pts


In [19]:
orderings.groupby(["league_key", "opens"]).points_vs_field.agg(
    ["count", "mean", "min", "max"]
).round(1)

count  mean   min   max
league_key opens                         
espn       RB        10   3.5  -8.6  18.2
           WR        10   0.9 -10.1  17.3
sleeper    RB        10 -31.1 -47.5  -9.0
           WR        10 -38.5 -52.8 -21.3

<a id="field"></a>
## 8. Who else is at the table

Every number so far assumes the other opponents draft straight off ADP and never deviate. That is
the cleanest way to ask "does departing from the market pay", and it is also the friendliest,
because a lone deviator competes with nobody for the position it is hoarding.

`field_model = 'mixed'` drops that assumption: every opponent gets its own randomly drawn opening
composition (drawn from the same 36 compositions this notebook ranks everywhere else, quarterback-
heavy ones included), three seeded replicates per draft. Whether that preserves each strategy's
relative order depends on the league now, and the two have pulled apart: Sleeper's rank correlation
between the passive and deviating fields is 0.98-0.99 either way — the quarterback effect is so
large it swamps whatever the field happens to be doing. ESPN's is only 0.61 under actual settings
(0.92 under the superflex counterfactual) — composition there is close enough to a coin flip that
who else is drafting actually reshuffles the ranking, not just its level.

**In Sleeper, the do-nothing ADP control does not improve against a deviating field — it stays
exactly where it was, 22nd of 37, in both.** That is a reversal from before the format change, when
a deviating field was the one thing that made pure ADP the best plan in this league. The most direct
reading: the mixed field now includes quarterback-heavy opponents too, so "the field deviates" no
longer means "the field only competes for RB/WR quotas and leaves quarterbacks sitting for whoever
holds ADP discipline" — some of that deviation is now competition for the same two quarterback slots
a Sleeper strategy actually wants, and a strategy holding zero of them doesn't benefit from it.

In ESPN, the shape from before mostly holds: pure ADP still climbs from a break-even 12th to a
positive-but-modest 21st of 37 against a deviating field (+9.9 pts), and a balanced opening
(`1QB1RB2WR1TE` now, rather than `2RB2WR1TE`) still leads the board either way.

The takeaway that survives in both leagues: a real draft room is closer to the mixed field than to
the ADP one, so the mixed-field ranking is the one to trust — but "the plan that wins once everyone
is deviating" now means very different plans in the two leagues (§5).

In [20]:
agreement = []
for league_key in ("sleeper", "espn"):
    for variant in ("actual", "superflex"):
        both = ranking(league_key, variant, "adp").merge(
            ranking(league_key, variant, "mixed"), on="strategy", suffixes=(" adp", " mixed")
        )
        agreement.append({
            "league": league_key, "variant": variant, "strategies": len(both),
            "rank correlation": both["points_vs_field adp"].corr(
                both["points_vs_field mixed"], method="spearman"
            ),
        })
pd.DataFrame(agreement).set_index(["league", "variant"]).round(3)

strategies  rank correlation
league  variant                                
sleeper actual             37             0.985
        superflex          37             0.982
espn    actual             37             0.608
        superflex          37             0.924

In [21]:
for league_key in ("sleeper", "espn"):
    table = ranking(league_key, "actual", "mixed")
    keep = pd.concat([
        table.head(5), table[table.strategy.isin(["ADP", "3RB2WR"])], table.tail(3)
    ]).drop_duplicates("strategy").sort_values("points_vs_field", ascending=False)
    print(f"\n=== {league_key} — against a field that also deviates (of {len(table)}) ===")
    print(keep.round(2).to_string())


=== sleeper — against a field that also deviates (of 37) ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1      2QB2RB1WR           104.47         5.51  12.12    54.33   11.78                11         11
2   2QB1RB1WR1TE           103.74         5.55  15.80    53.90   10.25                11         11
3      2QB1RB2WR            99.70         5.56  13.20    56.28   10.14                11         11
4      2QB2RB1TE            94.08         5.68  14.94    53.90    6.33                11         11
5         2QB3RB            83.42         5.77  11.26    51.08    5.18                10         11
22           ADP           -13.65         7.77   6.71    32.90   -1.05                 4         11
25        3RB2WR           -31.88         8.18   5.19    26.62   -1.98                 2         11
35        3RB2TE           -68.10         8.75   4.33    24.03   -3.81                 1         11
36        4WR1TE           -68.74     

In [22]:
# How far the do-nothing control moves once the rest of the room stops following ADP.
for league_key in ("sleeper", "espn"):
    moves = []
    for field_model in ("adp", "mixed"):
        table = ranking(league_key, "actual", field_model)
        moves.append((int(table.index[table.strategy == "ADP"][0]),
                      float(table.loc[table.strategy == "ADP", "points_vs_field"].iloc[0])))
    (adp_rank, adp_value), (mixed_rank, mixed_value) = moves
    print(f"{league_key:>8}: pure-ADP discipline ranks {adp_rank} vs a passive field "
          f"({adp_value:+.1f}) and {mixed_rank} vs a deviating one ({mixed_value:+.1f})")

 sleeper: pure-ADP discipline ranks 22 vs a passive field (-0.0) and 22 vs a deviating one (-13.7)
    espn: pure-ADP discipline ranks 12 vs a passive field (+0.0) and 21 vs a deviating one (+9.9)


<a id="superflex"></a>
## 9. Superflex: real now, not hypothetical

Section 1 already said it, but it's worth restating here because this used to be the section that
answered it: **Sleeper has a superflex slot for real now.** Everything in sections 2-8 already
reflects that — `variant = 'actual'` *is* the superflex answer for Sleeper, not a preview of one.

What's left for this section is narrower. `variant = 'superflex'` re-runs the sweep with one bench
spot converted into a *second* superflex slot, on top of the one Sleeper's actual settings already
have — draft length and roster size are unchanged, and the only thing that moves is one more slot's
worth of quarterback eligibility. For ESPN, which still has zero superflex slots, this variant is
exactly what it always was: the original one-slot counterfactual.

The two leagues' quarterback slopes, actual vs superflex, side by side, are in the table below.

Two things to read off it. First, ESPN's own counterfactual confirms the mechanism cleanly: its
quarterback slope goes from indistinguishable from zero (t = 0.06) to the largest positive effect on
that side of the table (t = 5.45, +34/pick) the moment a superflex slot exists — the same jump
Sleeper already made in real life. Second, Sleeper's slope was already enormous under its *actual*,
one-superflex-slot settings (+59/pick); a second superflex slot pushes it further (+100/pick), but
the marginal gain from that second slot (+41) is smaller than the gain from the first one already
baked into "actual" — the shape you'd expect, since the second slot is competing with everything
that already wants the first one.

The caveat from section 1 applies most to whichever number is furthest from a market that's already
repriced: ESPN's superflex figures and Sleeper's second-superflex figures are both upper bounds,
built on a one-quarterback ADP board pretending to be a two- or three-quarterback one. Sleeper's
*actual* row is the closest thing here to a fair test, and even that assumes the market hasn't fully
caught up to the league's own new rules yet.

In [23]:
for position in ("QB", "RB"):
    rows = []
    for league_key in ("sleeper", "espn"):
        for variant in ("actual", "superflex"):
            frame = compositions[
                (compositions.league_key == league_key)
                & (compositions.variant == variant)
            ]
            rows.append({"league": league_key, "variant": variant, **trend(frame, position)})
    print(f"\n=== value of the Nth {position} in the opening, actual vs superflex ===")
    print(pd.DataFrame(rows).drop(columns="position")
          .set_index(["league", "variant"]).round(2).to_string())


=== value of the Nth QB in the opening, actual vs superflex ===
                   0 in opening  1 in opening  2 in opening  3 in opening  4 in opening  5 in opening  pts/extra pick      t     p
league  variant                                                                                                                   
sleeper actual           -55.13         26.31         63.26           NaN           NaN           NaN           59.20   7.36  0.00
        superflex        -70.67         34.92        129.01           NaN           NaN           NaN           99.84  10.54  0.00
espn    actual           -14.79         -9.88        -14.30           NaN           NaN           NaN            0.24   0.06  0.96
        superflex        -27.22         18.79         39.81           NaN           NaN           NaN           33.51   5.45  0.00

=== value of the Nth RB in the opening, actual vs superflex ===
                   0 in opening  1 in opening  2 in opening  3 in opening  4 in open

In [24]:
for league_key in ("sleeper", "espn"):
    table = ranking(league_key, "superflex", "mixed")
    keep = pd.concat([table.head(5), table[table.strategy == "ADP"]]).drop_duplicates("strategy")
    print(f"\n=== {league_key} / superflex — best openings against a deviating field ===")
    print(keep.round(2).to_string())


=== sleeper / superflex — best openings against a deviating field ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1      2QB2RB1WR           145.96         5.06  15.80    61.26   13.16                11         11
2      2QB1RB2WR           141.15         5.23  17.32    57.58   10.91                11         11
3   2QB1RB1WR1TE           138.15         5.26  17.10    58.87   14.29                11         11
4         2QB3RB           128.62         5.32  14.07    58.44    7.34                11         11
5      2QB2RB1TE           126.37         5.37  16.02    55.84    8.65                11         11
22           ADP           -20.08         7.91   5.63    31.39   -1.29                 3         11

=== espn / superflex — best openings against a deviating field ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1      2QB2RB1WR            70.52         4.56  20.30    54.5

<a id="doing"></a>
## 10. What to actually do

Pulling the sections together, with each claim carrying the section that supports it. The two
leagues no longer share a strategy — that's the headline change since this was last written.

**Sleeper (14 teams, superflex):**
1. **Take two quarterbacks in the first five rounds.** This is the largest, most significant effect
   in the entire notebook (+59 pts per quarterback added to the opening, t = 7.36, positive in
   11/11 seasons — §6), and it holds up against a field that's also drafting aggressively, not just
   a passive one (§8). `2QB1RB2WR` is the best opening against a passive field; `2QB2RB1WR`
   overtakes it once the rest of the room is deviating too (§5, §8).
2. **Don't run the full house anyway.** Three backs in five rounds still ranks near the bottom —
   28th of 37 — worse than doing nothing at all (§5). Superflex didn't rescue that plan; it made the
   opportunity cost of not spending those picks on a second quarterback larger.
3. **If you're skipping the second quarterback, back still edges receiver on average** (+7 pts,
   opening RB vs opening WR — §7), but treat that as a mild tiebreaker, not a plan: it's dwarfed by
   the quarterback effect, and the strict "every RB-opener beats every WR-opener" version of the
   claim no longer holds (§7).
4. **The market hasn't caught up yet, and won't stay this generous.** These numbers are built on a
   one-quarterback consensus ADP board — real superflex ADP already prices quarterbacks higher than
   this. Read the *size* of the effect, not just its direction, as a moving target (§1, §9).

**ESPN (10 teams, no superflex) — unchanged:**
5. Composition still barely matters here: no position's slope clears significance, and the only
   things that reliably lose are the extreme, all-in openings (§6). `1QB1RB2WR1TE` is the best
   opening on the numbers but doesn't clear significance either (§5).
6. A field that's also deviating still makes the do-nothing ADP control look better than it does
   against a passive one (§8) — the same shape as before, just no single opening left to name as
   *the* plan for both leagues at once (§6).

**The two leagues no longer have a shared answer.** The one opening that used to clear significance
in both at once, `2RB2WR1TE`, doesn't exist any more — Sleeper's board is now organized entirely
around quarterback count, and nothing about ESPN's has moved (§6). If you're drafting in both
leagues this year, the Sleeper plan and the ESPN plan have to be two different plans.

The cell below assembles the per-league recommendation from the tables rather than from that list,
so if a rebuild moves the numbers it moves here too.

In [25]:
for league_key in ("sleeper", "espn"):
    settings = q("SELECT * FROM league_settings WHERE league_key = ?", [league_key]).iloc[0]
    passive = ranking(league_key, "actual", "adp")
    deviating = ranking(league_key, "actual", "mixed")
    best = passive.iloc[0]
    rb_first = orderings[(orderings.league_key == league_key) & (orderings.opens == "RB")]
    wr_first = orderings[(orderings.league_key == league_key) & (orderings.opens == "WR")]

    print(f"\n{'=' * 68}\n{league_key.upper()} — {settings.team_count} teams, "
          f"{settings.rec_pts} PPR, {settings.flex_slots} flex, "
          f"{settings.superflex_slots} superflex\n{'=' * 68}")
    print(f"  best opening vs a passive field : {best.strategy} "
          f"({best.points_vs_field:+.0f} pts, t={best.t_stat:.2f})")
    print(f"  best opening vs a deviating field: {deviating.iloc[0].strategy} "
          f"({deviating.iloc[0].points_vs_field:+.0f} pts, t={deviating.iloc[0].t_stat:.2f})")
    print(f"  full house (3RB2WR)             : rank {int(passive.index[passive.strategy == '3RB2WR'][0])}"
          f" of {len(passive)}")
    print(f"  open RB vs open WR              : "
          f"{rb_first.points_vs_field.mean() - wr_first.points_vs_field.mean():+.0f} pts for RB first")
    print(f"  QB in the first five rounds     : "
          f"{'yes — superflex' if settings.superflex_slots else 'no — one QB slot, no flex eligibility'}")


SLEEPER — 14 teams, 0.5 PPR, 1 flex, 1 superflex
  best opening vs a passive field : 2QB1RB2WR (+85 pts, t=5.75)
  best opening vs a deviating field: 2QB2RB1WR (+104 pts, t=11.78)
  full house (3RB2WR)             : rank 28 of 37
  open RB vs open WR              : +7 pts for RB first
  QB in the first five rounds     : yes — superflex

ESPN — 10 teams, 1.0 PPR, 1 flex, 0 superflex
  best opening vs a passive field : 1QB1RB2WR1TE (+18 pts, t=1.21)
  best opening vs a deviating field: 1QB1RB2WR1TE (+44 pts, t=2.50)
  full house (3RB2WR)             : rank 18 of 37
  open RB vs open WR              : +3 pts for RB first
  QB in the first five rounds     : no — one QB slot, no flex eligibility


### The 2026 board

Strategy decides *which position* to take; `draft_value` decides which player, by pricing this
year's projection against what that ADP slot has historically returned. `projected_surplus_rank` is
the draft-board column — at any given pick, who is expected to return the most over what he costs.

Shown per position so it can be read the way a draft actually goes: you are picking within a
position tier, not off one global list.

In [26]:
live_season = q("SELECT MAX(season) AS s FROM draft_value").s.iloc[0]
board = q("""
    SELECT position, player_name, ROUND(consensus_adp, 1) AS adp,
           ROUND(projected_surplus, 1) AS surplus, CAST(projected_surplus_rank AS INT) AS rank
    FROM draft_value
    WHERE league_key = 'sleeper' AND season = ? AND projected_surplus IS NOT NULL
      AND consensus_adp <= 120
    QUALIFY ROW_NUMBER() OVER (PARTITION BY position ORDER BY projected_surplus DESC) <= 8
    ORDER BY position, projected_surplus DESC
""", [int(live_season)])
print(f"{live_season} — best value per position inside the first 10 rounds (Sleeper scoring)")
board

2026 — best value per position inside the first 10 rounds (Sleeper scoring)


,position,player_name,adp,surplus,rank
0,QB,Jalen Hurts,74.9,45.7,1
1,QB,Bo Nix,119.1,10.5,31
2,QB,Josh Allen,23.8,7.8,35
3,QB,Jaxson Dart,101.9,3.2,40
4,QB,Caleb Williams,91.4,1.4,46
5,QB,Jayden Daniels,68.8,-2.2,98
6,QB,Justin Herbert,97.4,-5.8,155
7,QB,Patrick Mahomes,104.5,-11.3,203
8,RB,Jeremiyah Love,25.7,38.2,2
9,RB,De'Von Achane,10.1,29.9,8


---

## Adding to this notebook

The simulator lives in `src/gold/draft_strategy.py`, not here — its docstring carries the full
method and the caveats, and `python -m src.gold.draft_strategy` reprints the summary tables. This
notebook only reads `draft_strategy_results` / `draft_strategy_summary` and the models they were
built from.

`q()` opens a read-only connection, runs, and closes it — so nothing here can hold a lock that
blocks `scripts/build_warehouse.sh`, and nothing here can write to the warehouse. See
`notebooks/README.md` for the conventions.

Useful while exploring:

```python
tables("gold")
columns("draft_strategy_summary")
peek("draft_strategy_results")
```

Worth testing if this gets picked up again:

- **Keeper/dynasty rules.** Everything here is a redraft simulation.
- **In-season roster churn.** The simulation drafts and then freezes; it has no waiver wire, and a
  strategy that leaves you thin at a position is punished less here than in a real season.
- **Weekly lineups rather than season totals.** Teams are scored on the hindsight-best lineup, which
  is the same fiction for every strategy but flatters high-variance rosters slightly.